In [1]:
import pandas as pd
import numpy as np
import re

In [ ]:
# Load dataset bersih dari folder data/
df = pd.read_csv("../data/kost_putri_cleaned.csv")

print(f"Total data kos siap digunakan: {len(df)} baris")
df[['Nama Kos', 'Biaya Perbulan/semester', 'Alamat', 'Fasilitas Kos']].head()

Total data kos siap digunakan: 63 baris


,Nama Kos,Biaya Perbulan/semester,Alamat,Fasilitas Kos
0,Kost Putri Asri,Rp750.000-1.200.000/bulan\n(sesuai tipe kamar),"Puluhdadi No.416, Ngropoh, Caturtunggal, Kec. ...","Kasur, kamar mandi dalam, wifi-internet, listr..."
1,Kost Putri Pak Seno,800.000 (kipas angin)-\n1.000.000 (ac)/bulan,"Jl. Seturan No.1, Kledokan, Caturtunggal, Kec....","kamar mandi luar 4, wifi, listrik, kasur, lema..."
2,Kost Perum APH Seturan A.24,RP600.000-700.000/bulan\n(sesuai tipe kamar),"Perum APH Seturan Blok A.24 Kledokan, Caturtun...","kamar mandi luar 2, listrik, kasur, lemari pak..."
3,Graha Seturan Kost Putri,Rp850.000,"Jln. Seturan Raya, Kledokan, Caturtunggal, Dep...","kasur, meja kecil, lemari pakaian, kamar mandi..."
4,Kost Diajeng,Rp900.000/bulan\ndibayar pertigabulan,"Puluhdadi, Kledokan, Caturtunggal, Kec. Depok,...","kasur, lemari, wifi, kamar mandi dalam. listri..."


In [9]:
def parse_user_query(query_text):
    """
    Menganalisis teks pertanyaan pengguna untuk mengekstrak kriteria:
    - Busget maksimum (harga)
    - Fasilitas yang diminta (AC, KM dalam, Wifi, Listrik)
    - Kata kunci area/lokasi
    """
    text = query_text.lower()
    criteria  = {
        'max_budget': None,
        'facilities': {},
        'location': None
    }

    #ekstraksi angka budget (detgeksi pola juta, jt, dll)
    juta_match = re.search(r'(\d+[\.,]?\d*)\s*(juta|jt)', text)
    if juta_match:
        val = float(juta_match.group(1).replace(',', '.'))
        criteria['max_budget'] = int(val * 1000000)
    else:
        #deteksi pola ribu
        rb_match = re.search(r'(\d+)|s*(rb|ribu)', text)
        if rb_match:
            criteria['max_budget'] = int(rb_match.group(1)) * 1000
        else:
            #deteksi angka murni, min 6 digit
            raw_nums = re.findall(r'\d{6,}', text.replace('.', ''))
            if raw_nums:
                criteria['max_budget'] = int(raw_nums[0])

    #ekstraksi fasilitas
    if re.search(r'\bac\b|air conditioner', text):
        criteria['facilities']['has_ac'] = True
    if re.search(r'km dalam|kamar mandi dalam|kamar mandi', text):
        criteria['facilities']['has_km_dalam'] = True
    if re.search(r'wifi|wi-fi|internet', text):
        criteria['facilities']['has_wifi'] = True
    if re.search(r'listrik', text):
        criteria['facilities']['has_listrik'] = True

    #ekstraksi lokasi
    known_areas = ['kledokan', 'seturan', 'ngropoh', 'condongcatur', 'puluhdadi', 'dabag', 'upn', 'ykpn']
    for area in known_areas:
        if area in text:
            criteria['location'] = area
            break
    
    return criteria

#teks fungsi ekstraksi
contoh_tanya = "Halo kak, ada kos di daerah Kledokan yang ada AC dan wifi budget sekitar 1.2 juta?"
print("Hasil Ekstraksi Entitas:")
print(parse_user_query(contoh_tanya))


Hasil Ekstraksi Entitas:
{'max_budget': 1200000, 'facilities': {'has_ac': True, 'has_wifi': True}, 'location': 'kledokan'}


In [10]:
def chat_assistant(user_message, df_source, top_n=3):
    """
    Menerima pertanyaan dalam bahasa sehari-hari,
    mencari kos yang relevan, dan mengembalikan teks balasan asisten.
    """
    # 1. Parsing input user
    criteria = parse_user_query(user_message)
    max_budget = criteria['max_budget']
    facs = criteria['facilities']
    loc = criteria['location']
    
    # 2. Filtering dataset
    candidate_df = df_source.copy()
    
    # Filter Lokasi jika ada
    if loc:
        candidate_df = candidate_df[
            candidate_df['Alamat'].str.lower().str.contains(loc, na=False) |
            candidate_df['Nama Kos'].str.lower().str.contains(loc, na=False)
        ]
        
    # Filter Fasilitas yang diminta
    for fac_col, required in facs.items():
        if required:
            candidate_df = candidate_df[candidate_df[fac_col] == True]
            
    # Filter Budget jika ada
    if max_budget:
        candidate_df = candidate_df[
            (candidate_df['harga_min'].isna()) | (candidate_df['harga_min'] <= max_budget)
        ]

    # Urutkan berdasarkan harga termurah jika data harga ada
    candidate_df = candidate_df.sort_values(by='harga_min', ascending=True, na_position='last')

    # 3. Susun Kalimat Respon (Generator)
    if candidate_df.empty:
        # Fallback jika tidak ada yang cocok sempurna
        return (
            f"Maaf kak, belum ada kos yang 100% cocok dengan kriteria tersebut "
            f"(Budget: Rp{max_budget:, if max_budget else '-'}, Fasilitas: {list(facs.keys())}, Lokasi: {loc}).\n\n"
            f"💡 *Saran:* Coba naikkan sedikit rentang budget atau kurangi salah satu fasilitas wajib."
        )

    # Header Respon
    filter_notes = []
    if loc: filter_notes.append(f"di sekitar **{loc.title()}**")
    if facs: 
        fac_labels = [k.replace('has_', '').replace('_', ' ').upper() for k in facs.keys()]
        filter_notes.append(f"fasilitas **{', '.join(fac_labels)}**")
    if max_budget: filter_notes.append(f"budget maksimal **Rp {max_budget:,}**".replace(",", "."))
    
    catatan_pencarian = ", ".join(filter_notes) if filter_notes else "semua kriteria"
    
    response = [f"Halo kak! Berdasarkan pencarian untuk {catatan_pencarian}, ini ada **{min(len(candidate_df), top_n)} pilihan kos terbaik**:\n"]

    # Ambil Top-N kos
    for i, (_, row) in enumerate(candidate_df.head(top_n).iterrows(), start=1):
        if pd.notna(row['harga_min']):
            if row['harga_min'] == row['harga_max']:
                harga_txt = f"Rp {int(row['harga_min']):,} / bulan".replace(",", ".")
            else:
                harga_txt = f"Rp {int(row['harga_min']):,} - {int(row['harga_max']):,} / bulan".replace(",", ".")
        else:
            harga_txt = "Hubungi pemilik untuk harga"

        wa_link = f"[Chat WhatsApp]({row['wa_url']})" if pd.notna(row.get('wa_url')) else "Kontak belum tersedia"
        maps_link = f"[Buka Google Maps]({row['maps_url']})" if pd.notna(row.get('maps_url')) else ""

        response.append(
            f"**{i}. {row['Nama Kos']}**\n"
            f"- 💰 **Harga:** {harga_txt}\n"
            f"- 📍 **Alamat:** {row['Alamat'][:60]}...\n"
            f"- ✨ **Fasilitas:** {row['Fasilitas Kos']}\n"
            f"- 📲 **Aksi:** {wa_link} | {maps_link}\n"
        )

    response.append("Ada lagi kriteria kos lain yang ingin kakak cari?")
    return "\n".join(response)

In [11]:
# UJI COBA 1: Pertanyaan dengan budget & AC & lokasi
pertanyaan_1 = "Ada kos di Kledokan yang ada AC budget 1.2 juta?"
print("=== PERTANYAAN 1 ===")
print(chat_assistant(pertanyaan_1, df))
print("\n" + "="*50 + "\n")

# UJI COBA 2: Pertanyaan santai hanya cari yang murah ada wifi di Seturan
pertanyaan_2 = "Cari kos putri di seturan yang ada wifinya dong budget 900rb"
print("=== PERTANYAAN 2 ===")
print(chat_assistant(pertanyaan_2, df))

=== PERTANYAAN 1 ===
Halo kak! Berdasarkan pencarian untuk di sekitar **Kledokan**, fasilitas **AC**, budget maksimal **Rp 1.200.000**, ini ada **3 pilihan kos terbaik**:

**1. Kost Putri Pak Seno**
- 💰 **Harga:** Rp 800.000 - 1.000.000 / bulan
- 📍 **Alamat:** Jl. Seturan No.1, Kledokan, Caturtunggal, Kec. Depok, Kabupa...
- ✨ **Fasilitas:** kamar mandi luar 4, wifi, listrik, kasur, lemari pakaian, meja, kursi, kipas angin/ac, dapur bersama, kulkas bersama
- 📲 **Aksi:** [Chat WhatsApp](https://wa.me/6285228622686) | [Buka Google Maps](https://www.google.com/maps/search/?api=1&query=Kos+Putri+Pak+Seno)

**2. Kost Baiti Jannti**
- 💰 **Harga:** Rp 900.000 / bulan
- 📍 **Alamat:** Jl. Seturan II No.76, Kledokan, Caturtunggal, Kec. Depok, Ka...
- ✨ **Fasilitas:** kasur, bantal guling sprei, lemari, meja kursi, kipas dinding, tempat sampah, dapur bersama, kulkas bersama, tempat parkir luas, listrik, wifi
- 📲 **Aksi:** [Chat WhatsApp](https://wa.me/6285329443778) | [Buka Google Maps](https://w